In [1]:
# ============================================================
# DÍA 4 — Limpieza de Datos
# Programa Data Analytics — 90 días
# ============================================================

import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset cargado")
print(f"Forma: {df.shape}")

Dataset cargado
Forma: (891, 12)


In [2]:
# ============================================================
# BLOQUE 1: Diagnóstico de valores nulos
# ============================================================

# Paso 1: ¿Cuántos nulos hay en cada columna?
print("=== Nulos por columna ===")
print(df.isnull().sum())
print()

# Paso 2: Porcentaje de nulos — más útil que el conteo puro
print("=== Porcentaje de nulos ===")
pct_nulos = (df.isnull().sum() / len(df) * 100).round(2)
print(pct_nulos)
print()

# Paso 3: Resumen ejecutivo — solo columnas que tienen al menos 1 nulo
print("=== Columnas con datos faltantes ===")
nulos = df.isnull().sum()
columnas_con_nulos = nulos[nulos > 0]
print(columnas_con_nulos)

=== Nulos por columna ===
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

=== Porcentaje de nulos ===
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64

=== Columnas con datos faltantes ===
Age         177
Cabin       687
Embarked      2
dtype: int64


In [3]:
# ============================================================
# BLOQUE 2: Tratar valores nulos
# ============================================================

# --- Caso 1: Columna Cabin — 77% de nulos
# Decisión de negocio: demasiados faltantes, no es útil
# La eliminamos

df_limpio = df.drop(columns=['Cabin'])
print(f"Columnas antes: {df.shape[1]}")
print(f"Columnas después de eliminar Cabin: {df_limpio.shape[1]}")
print()

# --- Caso 2: Columna Age — 20% de nulos
# Decisión: imputar con la mediana (no la media, porque la edad tiene outliers)
# NOTA: recordar la lección del Día 1 — mediana ≠ midrange

mediana_edad = df_limpio['Age'].median()
print(f"Mediana de edad: {mediana_edad}")

df_limpio['Age'] = df_limpio['Age'].fillna(mediana_edad)
print(f"Nulos en Age después de fillna: {df_limpio['Age'].isnull().sum()}")
print()

# --- Caso 3: Columna Embarked — solo 2 nulos
# Decisión: imputar con la moda (valor más frecuente)

moda_embarked = df_limpio['Embarked'].mode()[0]
print(f"Moda de Embarked: {moda_embarked}")

df_limpio['Embarked'] = df_limpio['Embarked'].fillna(moda_embarked)
print(f"Nulos en Embarked después de fillna: {df_limpio['Embarked'].isnull().sum()}")

Columnas antes: 12
Columnas después de eliminar Cabin: 11

Mediana de edad: 28.0
Nulos en Age después de fillna: 0

Moda de Embarked: S
Nulos en Embarked después de fillna: 0


In [4]:
# ============================================================
# BLOQUE 3: Detectar y eliminar duplicados
# ============================================================

# ¿Hay filas completamente duplicadas?
print(f"Filas duplicadas: {df_limpio.duplicated().sum()}")

# Cómo se vería si hubiera duplicados — ejemplo simulado
df_con_dup = pd.concat([df_limpio.head(3), df_limpio.head(3)], ignore_index=True)
print(f"\nDataFrame con duplicados simulados: {df_con_dup.shape[0]} filas")
print(f"Duplicados detectados: {df_con_dup.duplicated().sum()}")

# Eliminar duplicados
df_sin_dup = df_con_dup.drop_duplicates()
print(f"Después de drop_duplicates(): {df_sin_dup.shape[0]} filas")

Filas duplicadas: 0

DataFrame con duplicados simulados: 6 filas
Duplicados detectados: 3
Después de drop_duplicates(): 3 filas


In [5]:
# ============================================================
# BLOQUE 4: Revisar y corregir tipos de datos
# ============================================================

# Ver los tipos actuales
print("=== Tipos de datos actuales ===")
print(df_limpio.dtypes)
print()

# Survived y Pclass son int, pero conceptualmente son categorías
# En análisis, a veces conviene tratarlas como tal

df_limpio['Survived'] = df_limpio['Survived'].astype(str).map({'0': 'No', '1': 'Sí'})
print("Survived convertido:")
print(df_limpio['Survived'].value_counts())
print()

# Revertimos para no romper análisis futuros
df_limpio['Survived'] = df_limpio['Survived'].map({'No': 0, 'Sí': 1})

# Ejemplo: crear una columna de edad como categoría
df_limpio['GrupoEdad'] = pd.cut(
    df_limpio['Age'],
    bins=[0, 12, 17, 60, 100],
    labels=['Niño', 'Adolescente', 'Adulto', 'Mayor']
)
print("=== Distribución por grupo de edad ===")
print(df_limpio['GrupoEdad'].value_counts())

=== Tipos de datos actuales ===
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked        object
dtype: object

Survived convertido:
Survived
No    549
Sí    342
Name: count, dtype: int64

=== Distribución por grupo de edad ===
GrupoEdad
Adulto         756
Niño            69
Adolescente     44
Mayor           22
Name: count, dtype: int64


In [6]:
# ============================================================
# BLOQUE 5: Verificación — el dataset está listo
# ============================================================

print("=== Estado final del dataset limpio ===")
print(f"Filas: {df_limpio.shape[0]}")
print(f"Columnas: {df_limpio.shape[1]}")
print()
print("Nulos restantes:")
print(df_limpio.isnull().sum()[df_limpio.isnull().sum() > 0])
print()
print("Tipos de datos:")
print(df_limpio.dtypes)

=== Estado final del dataset limpio ===
Filas: 891
Columnas: 12

Nulos restantes:
Series([], dtype: int64)

Tipos de datos:
PassengerId       int64
Survived          int64
Pclass            int64
Name             object
Sex              object
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Embarked         object
GrupoEdad      category
dtype: object


In [32]:
# Ejercicio 1 — Diagnóstico
#  Carga el dataset, genera el resumen de nulos (conteo y porcentaje) y responde:
# ¿qué columnas tienen más del 10% de datos faltantes?

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset cargado")
print(f"Forma: {df.shape}")

print("=== Nulos por cada columna ===")
nulos = df.isnull().sum()
print(nulos)

print("=== Porcentaje de nulos en cada columna ===")
pct_nulos = (df.isnull().sum() / len(df) * 100).round(2)
print(pct_nulos)

print(" === Respuesta Ejercicio 1 ===")
respuesta_1 = "La columna de Age cuenta con un 19.87% y Cabin con un 77.1% de sus datos nulos"
print(respuesta_1)

# Ejercicio 2 — Decisión de imputación
#  Imputa los nulos de Age usando la media en lugar de la mediana. Luego responde:
# ¿cuánto difiere el resultado del valor que usamos en clase (mediana = 28.0)?
# ¿Por qué la mediana es generalmente preferible aquí?

media_edad = df['Age'].mean().round(2)
print(f"Media de edad: {media_edad}")

df_limpio['Age'] = df_limpio['Age'].fillna(media_edad)
print(f"Nulos en Age modificanos por la media: {df_limpio['Age'].isnull().sum()}")
respuesta_2 = "Para evitar outliders ocasionado por valores muy extremos y tener un promedio muy distinto al real."
print(respuesta_2)

#Ejercicio 3 — Segmentación con pd.cut()
#  Usando la columna Age ya limpia, crea una columna RangoPrecio que clasifique la
# tarifa Fare en tres categorías: Bajo (0–20), Medio (20–100), Alto (100+).
# Luego muestra cuántos pasajeros hay en cada rango.

df_limpio['RangoPrecio'] = pd.cut(
    df_limpio['Fare'],
    bins=[0, 20, 100, float('inf')],
    labels=['Bajo', 'Medio', 'Alto'],
    include_lowest= True
)
print("=== Tarifa por rango ===")
print(df_limpio['RangoPrecio'].value_counts())

# Ejercicio 4 — Análisis combinado
#  Con el dataset limpio, agrupa por GrupoEdad y calcula la tasa de supervivencia
# de cada grupo. ¿Qué grupo tuvo mayor probabilidad de sobrevivir?

df_limpio['GrupoEdad'] = pd.cut(
    df_limpio['Age'],
    bins=[0, 12, 17, 60, 100],
    labels=['Niño', 'Adolescente', 'Adulto', 'Mayor']
)

tasa_supervivencia = df_limpio.groupby('GrupoEdad', observed= True)['Survived'].mean().round(2)
print("=== Tasa de supervivencia por grupo de edad")
print(tasa_supervivencia)
respuesta_4 = "Dado los resultados de supervivencia por categoria de edad, los niños con un 58% tienen la tasa de supervivencia mas alta, seguidos por los adolecentes con un 48%"

# Ejercicio 5 — Reflexión profesional (sin código)

# Imagina que eres analista en una aerolínea y recibes un dataset de vuelos con
# 30% de nulos en la columna delay_minutes. ¿Eliminarías esa columna,
# eliminarías las filas, o imputarías? ¿Con qué valor imputarías y por qué?

respuesta_5 = (
    "Dada el contexto de la aerolinea,"
    " imputaria los valores usando el promedio de la columna, los minutos de retraso"
    " pueden variar pero los extremos de los valores no deberian ser enormes como para considerarlos outliders"
)
print(respuesta_5)

Dataset cargado
Forma: (891, 12)
=== Nulos por cada columna ===
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
=== Porcentaje de nulos en cada columna ===
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64
 === Respuesta Ejercicio 1 ===
La columna de Age cuenta con un 19.87% y Cabin con un 77.1% de sus datos nulos
Media de edad: 29.7
Nulos en Age modificanos por la media: 0
Para evitar outliders ocasionado por valores muy extremos y tener un promedio muy distinto al real.
=== Tarifa por rango ===
RangoPrecio
Bajo     515
Medio    323
Alto      53
Name: count, dtype: int64
=== Tasa de